<h1>Sprint 2: Uso de grafos en redes sociales</h1>
Autor: Adrián Robles Arques

In [1]:
# Vamos a importar las librerias necesarias
import numpy as np
import matplotlib.pyplot as plt
import pyspark as ps
import pyspark.sql.functions as F
import networkx as nx
from pyspark.sql import DataFrame

# Creamos una sesion de Spark
spark = ps.sql.SparkSession.builder \
    .appName("Grafos en Spark") \
    .master("local[*]") \
    .getOrCreate()

In [2]:
# Leemos los datos de los archivos CSV
edges = spark.read.csv("rednova_edge_list.csv", header=True, inferSchema=True)
data = spark.read.csv("rednova_usuarios.csv", header=True, inferSchema=True)

In [4]:
# Mostramos datos de los usuarios
data.show(10)

+----------+---------------+----------+--------------------+---------+--------------------+-------------+-------------+-----------+-----------+-------+--------------+
|ID Usuario| Nombre Usuario|Seguidores|    ID de Seguidores|Siguiendo|     ID de Siguiendo|Publicaciones|Likes Totales|Comentarios|Impresiones|Alcance|Engagement (%)|
+----------+---------------+----------+--------------------+---------+--------------------+-------------+-------------+-----------+-----------+-------+--------------+
|         1|        @cperez|        27|[7047, 1995, 7693...|       35|[4204, 5973, 3600...|          156|            1|          0|         19|     15|          5.38|
|         2|      @okrueger|        90|[4698, 9728, 9199...|      307|[495, 5665, 7371,...|          281|            4|          1|         30|     18|           6.4|
|         3| @yolandasnyder|       132|[9936, 9348, 3919...|       64|[9526, 6242, 3569...|          590|           12|          1|        151|     97|          9.92

In [5]:
# Mostramos los datos de las aristas
edges.show(10)

+------+------+
|source|target|
+------+------+
|  7047|     1|
|  1995|     1|
|  7693|     1|
|  6462|     1|
|  3059|     1|
|  8172|     1|
|  2364|     1|
|  4667|     1|
|  8062|     1|
|  2606|     1|
+------+------+
only showing top 10 rows



In [ ]:
# Generamos el gráfico de grafos a partir de los datos

def create_graph(edges: DataFrame, data: DataFrame) -> DataFrame:
    return edges.join(data[['ID Usuario', "Nombre Usuario"]], edges["source"] == data["ID Usuario"], "left") \
                .select("source", "target", "Nombre Usuario")
graph = create_graph(edges, data)

# Visualizamos el gráfico

def visualize_graph(graph: DataFrame):
    # Convertimos el DataFrame de Spark a un DataFrame de Pandas
    graph_pd = graph.toPandas()

    # Obtenemos los datos de seguidores de cada usuario
    seguidores_pd = data.select("ID Usuario", "Seguidores").toPandas()
    id_to_seguidores = dict(seguidores_pd.values)

    # Creamos un diccionario para mapear IDs a nombres de usuario
    id_to_name = dict(graph_pd[["source", "Nombre Usuario"]].values)

    # Creamos el grafo usando las IDs
    G = nx.from_pandas_edgelist(graph_pd, source="source", target="target")
    
    # Establecemos los colores de los nodos basados en el número de seguidores
    node_colors = [id_to_seguidores.get(node, 0) for node in G.nodes()]
    
    # Definimos tamaños de los nodos basados en el número de seguidores
    node_sizes = [100 + 10 * id_to_seguidores.get(node, 0) for node in G.nodes()]

    # Normalizamos los colores para el gradiente
    import matplotlib.cm as cm
    min_color, max_color = min(node_colors), max(node_colors)
    norm = plt.Normalize(min_color, max_color)
    cmap = cm.viridis

    # Dibujamos el grafo con gradiente de colores y el tamaño dependiendo del número de seguidores
    fig, ax = plt.subplots(figsize=(30, 30))
    pos = nx.circular_layout(G) # Usamos un layout circular para una mejor visualización
    
    # Dibujamos los nodos con el gradiente de colores y tamaños
    nx.draw_networkx_nodes(
        G, pos, ax= ax, node_size=node_sizes, node_color=node_colors, cmap=cmap,
        vmin=min_color, vmax=max_color, alpha= 0.9, 
    )
    
    #Dibujamos las aristas con flechas 
    nx.draw_networkx_edges(G, pos, ax= ax, alpha=0.5, arrowstyle='->', arrowsize=10)
    
    # Establecemos las etiquetas de los nodos
    nx.draw_networkx_labels(G, pos, ax= ax, labels=id_to_name, font_size=10, font_color='black', font_family='sans-serif')
    
    # Introducción del ScalarMappable
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm) 
    sm.set_array([])
    
    fig.colorbar(sm, ax = ax, label='Número de Seguidores')
    fig.suptitle("Grafo de Usuarios (Gradiente por Seguidores)")
    fig.show()

In [ ]:
# Mostramos el gráfico
graph.show(10)

# Ejecutamos la visualización del grafo
visualize_graph(graph)

+------+------+----------------+
|source|target|  Nombre Usuario|
+------+------+----------------+
|  7047|     1|       @samuel09|
|  1995|     1|  @andrewsummers|
|  7693|     1|      @deborah54|
|  6462|     1|    @erichawkins|
|  3059|     1| @olsonstephanie|
|  8172|     1|   @kellyjackson|
|  2364|     1| @matthewwebster|
|  4667|     1|    @christine01|
|  8062|     1|@cliffordramirez|
|  2606|     1|    @danielstone|
+------+------+----------------+
only showing top 10 rows



C:\Users\demad\AppData\Local\Temp\ipykernel_24580\786960080.py:50: UserWarning: 

The arrowstyle keyword argument is not applicable when drawing edges
with LineCollection.

To make this warning go away, either specify `arrows=True` to
force FancyArrowPatches or use the default values.
Note that using FancyArrowPatches may be slow for large graphs.

  nx.draw_networkx_edges(G, pos, alpha=0.5, arrowstyle='->', arrowsize=10)


ValueError: Unable to determine Axes to steal space for Colorbar. Either provide the *cax* argument to use as the Axes for the Colorbar, provide the *ax* argument to steal space from it, or add *mappable* to an Axes.

Error in callback <function _draw_all_if_interactive at 0x00000155755A9B20> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 

In [ ]:
# Vamos a analizar los datos empleando DataFrames de Spark

data_df = data.toDataFrame() # Nos aseguramos de que data es un DataFrame de Spark
